# A_01 — Text EDA (Track A: normalisation + blocking)

Goal: find the noise patterns and the evidence that true matches share, to design `normalize.py` and `blocking.py`.
Data: the 5% dev sample (`sample_5pct`) for train statistics, a 5% id-bucket slice of test for France.
Conclusions are written up in `docs/method/blocking.md`.

Prerequisites: `python -m src.io_utils`, `python -m src.sample`, `python -m src.stub` (run from `code/business_entity_resolution/`).

In [ ]:
import sys, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import polars as pl
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist

sys.path.insert(0, str(Path.cwd().parent / "code" / "business_entity_resolution"))
from src import contracts as C
from src.stub import stub_normalize

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(60); pl.Config.set_tbl_width_chars(220)
R = {s: C.load_records("sample_5pct", s) for s in (1, 2, 3)}            # raw sample records
N = {s: C.load_norm("sample_5pct", s) for s in (1, 2, 3)}               # stub-normalised (lowercase ascii)
T = {s: C.load_records("test", s).filter(pl.col("entity_id").str.slice(3).cast(pl.Int64) % 100 < 5) for s in (1, 2, 3)}
gt_s1, gt = C.load_ground_truth("sample_5pct")
others_raw, others_norm = pl.concat([R[2], R[3]]), pl.concat([N[2], N[3]])
{f"S{s}": d.height for s, d in R.items()}

## 1. Scripts
Share of records containing each Indic script (India only; US and train S1 are pure ASCII).

In [ ]:
SCRIPTS = {"Deva": "\u0900-\u097F", "Beng": "\u0980-\u09FF", "Guru": "\u0A00-\u0A7F", "Gujr": "\u0A80-\u0AFF",
           "Orya": "\u0B00-\u0B7F", "Taml": "\u0B80-\u0BFF", "Telu": "\u0C00-\u0C7F", "Knda": "\u0C80-\u0CFF", "Mlym": "\u0D00-\u0D7F"}
rows = []
for s in (1, 2, 3):
    d = R[s].filter(pl.col("country") == "India")
    for col in ("business_name", "business_address"):
        x = d[col]
        rows.append({"src": f"S{s}", "field": col[9:], "non_ascii_%": 100 * x.str.contains(r"[^\x00-\x7F]").mean(),
                     "any_indic_%": 100 * x.str.contains("[\u0900-\u0DFF]").mean(),
                     **{k: 100 * x.str.contains(f"[{v}]").mean() for k, v in SCRIPTS.items()}})
pl.DataFrame(rows).with_columns(pl.selectors.float().round(2))

In [ ]:
# Native script in addresses is (almost always) only the state name at the end
R[2].filter(pl.col("business_address").str.contains("[\u0900-\u0DFF]"))["business_address"].sample(6, seed=1).to_list()

In [ ]:
# Native-script names are phonetic renderings of English words; unidecode output shows the pattern
d = R[2].filter(pl.col("business_name").str.contains("[\u0900-\u0DFF]"))
pl.DataFrame({"raw": d["business_name"].head(8), "unidecode (stub)": N[2].join(d.select("entity_id"), on="entity_id", how="semi")["name_clean"].head(8)})

In [ ]:
# Other non-ASCII: injected accents, zero-width joiners, cp1252 mojibake
cnt = Counter(ch for s in (2, 3) for x in R[s]["business_name"].to_list() + R[s]["business_address"].to_list()
              for ch in x if ord(ch) > 127 and not 0x900 <= ord(ch) <= 0xDFF)
[(c, unicodedata.name(c, "?"), n) for c, n in cnt.most_common(15)]

## 2. Name noise

In [ ]:
for s in (1, 2, 3):
    n = R[s]["business_name"]
    lead = n.str.extract(r"^([^\p{L}\p{N}]+)").drop_nulls().str.strip_chars()
    low = n.str.to_lowercase()
    print(f"S{s}: leading junk {lead.len() / n.len():.3f} {lead.value_counts(sort=True).head(8).rows()}")
    print(f"     ends with ']' {n.str.ends_with(']').mean():.3f}  '(' {n.str.contains(r'\(').mean():.3f}  domain {low.str.contains(r'\.(com|in|net|org|co|io|biz)\b').mean():.3f}"
          f"  dba {low.str.contains(r'\b(dba|d/b/a)\b').mean():.4f}  aka {low.str.contains(r'\baka\b').mean():.4f}  ALLCAPS {(n == n.str.to_uppercase()).mean():.3f}")

In [ ]:
# How these look next to the S1 name of the same entity
pairs = (gt.join(R[1].select(pl.col("entity_id").alias("s1_id"), pl.col("business_name").alias("s1_name")), on="s1_id")
           .join(others_raw.rename({"entity_id": "id"}), on="id"))
def show(cond, n=6):
    return pairs.filter(cond).select("s1_name", "business_name").sample(n, seed=3)
show(pl.col("business_name").str.ends_with("]"))

In [ ]:
show(pl.col("business_name").str.to_lowercase().str.contains(r"\.(com|in|net|org)\b"))

In [ ]:
show(pl.col("business_name").str.to_lowercase().str.contains(r"\b(dba|aka)\b"))

## 3. Legal forms and honorifics (first / last tokens)

In [ ]:
def edge_tokens(names, which):
    t = names.str.split(" ")
    t = t.list.first() if which == "first" else t.list.last()
    return [f"{a}:{b}" for a, b in t.value_counts(sort=True).head(15).rows()]
for s in (1, 2, 3):
    for ctry in ("US", "India"):
        x = N[s].filter(pl.col("country") == ctry)["name_clean"]
        print(f"S{s} {ctry:5s} first: {edge_tokens(x, 'first')}\n{'':9s} last : {edge_tokens(x, 'last')}")
tf = stub_normalize(T[1].filter(pl.col("country") == "France"))["name_clean"]
print(f"test S1 France first: {edge_tokens(tf, 'first')}\n{'':14s}last : {edge_tokens(tf, 'last')}")

## 4. Address anatomy
Postcodes are essentially absent; 5-digit numbers in US addresses are house numbers.

In [ ]:
def addr_stats(df, tag):
    a = df["business_address"]; low = a.str.to_lowercase()
    return {"set": tag, "n": df.height, "empty": (a == "").mean(), "5-digit": a.str.contains(r"\b\d{5}\b").mean(),
            "6-digit": a.str.contains(r"\b\d{6}\b").mean(), "null-token": low.str.contains(r"\b(null|n/a)\b|<null>").mean(),
            "near/opp": low.str.contains(r"\b(near|nr|opp|opposite|behind)\b").mean(), "commas": a.str.count_matches(",").mean()}
rows = [addr_stats(R[s].filter(pl.col("country") == c), f"train S{s} {c}") for s in (1, 2, 3) for c in ("US", "India")]
rows += [addr_stats(T[s].filter(pl.col("country") == "France"), f"test S{s} France") for s in (1, 2, 3)]
pl.DataFrame(rows).with_columns(pl.selectors.float().round(3))

In [ ]:
for s in (1, 2, 3):
    for c in ("US", "India"):
        print(f"S{s} {c}:", *R[s].filter((pl.col("country") == c) & (pl.col("business_address") != ""))["business_address"].sample(3, seed=5).to_list(), sep="\n    ")

## 5. What true matches share (vs random same-country pairs)
Computed on stub-normalised text; tokens shorter than 3 characters ignored.

In [ ]:
NULLS = r"\b(null|n a|na|none)\b"
tok = lambda c: pl.col(c).str.split(" ").list.eval(pl.element().filter(pl.element().str.len_chars() > 2))

def agreement(pairs):
    j = (pairs.join(N[1].select(pl.col("entity_id").alias("s1_id"), pl.col("name_clean").alias("n1"), pl.col("addr_clean").alias("a1")), on="s1_id")
              .join(others_norm.select(pl.col("entity_id").alias("id"), pl.col("name_clean").alias("n2"), pl.col("addr_clean").alias("a2")), on="id")
              .with_columns(pl.col("a1", "a2").str.replace_all(NULLS, ""))
              .with_columns(tok("n1").alias("t1"), tok("n2").alias("t2"), tok("a1").alias("u1"), tok("a2").alias("u2"),
                            pl.col("a1").str.extract(r"(\d+)").alias("h1"), pl.col("a2").str.extract(r"(\d+)").alias("h2")))
    both_num = j.filter(pl.col("h1").is_not_null() & pl.col("h2").is_not_null())
    sim = lambda a, b, sc=fuzz.token_set_ratio: np.median(cpdist(j[a].to_list(), j[b].to_list(), scorer=sc, workers=-1))
    return {
        "name shares a token": j.select((pl.col("t1").list.set_intersection("t2").list.len() > 0).mean()).item(),
        "name first token equal": j.select((pl.col("t1").list.first() == pl.col("t2").list.first()).mean()).item(),
        "address shares a token": j.select((pl.col("u1").list.set_intersection("u2").list.len() > 0).mean()).item(),
        "first number equal (both have one)": both_num.select((pl.col("h1") == pl.col("h2")).mean()).item(),
        "median name token_set_ratio": sim("n1", "n2"),
        "median address token_set_ratio": sim("a1", "a2"),
    }

neg = pl.concat([pl.DataFrame({"s1_id": N[1].filter(pl.col("country") == c)["entity_id"].sample(30000, with_replacement=True, seed=1),
                               "id": others_norm.filter(pl.col("country") == c)["entity_id"].sample(30000, with_replacement=True, seed=2)})
                 for c in ("US", "India")])
m, r = agreement(gt.sample(60000, seed=1)), agreement(neg)
pl.DataFrame({"evidence": list(m), "true matches": list(m.values()), "random pairs": list(r.values())})

In [ ]:
# The hard 14%: true matches sharing no name token (>2 chars)
j = (gt.join(N[1].select(pl.col("entity_id").alias("s1_id"), pl.col("name_clean").alias("n1"), "country"), on="s1_id")
       .join(others_norm.select(pl.col("entity_id").alias("id"), pl.col("name_clean").alias("n2"), "is_native_script"), on="id")
       .with_columns(tok("n1").alias("t1"), tok("n2").alias("t2")))
hard = j.filter(pl.col("t1").list.set_intersection("t2").list.len() == 0)
print(f"{hard.height:,} of {j.height:,} ({hard.height / j.height:.1%}); native script among them {hard['is_native_script'].mean():.1%} "
      f"(overall {j['is_native_script'].mean():.1%}); by country {hard.group_by('country').len().rows()}")
(hard.filter(~pl.col("is_native_script")).select("s1_id", "id")
     .join(R[1].select(pl.col("entity_id").alias("s1_id"), pl.col("business_name").alias("S1 name"), pl.col("business_address").alias("S1 addr")), on="s1_id")
     .join(others_raw.select(pl.col("entity_id").alias("id"), pl.col("business_name").alias("S2/3 name"), pl.col("business_address").alias("S2/3 addr")), on="id")
     .drop("s1_id", "id").sample(10, seed=7))

In [ ]:
# Match multiplicity: most S1s have records in both sources
g = gt.with_columns(pl.col("id").str.slice(0, 2).alias("src")).group_by("s1_id").agg(
    (pl.col("src") == "S2").sum().alias("n_s2"), (pl.col("src") == "S3").sum().alias("n_s3"))
print("S1 with both S2 and S3 matches:", round(((g["n_s2"] > 0) & (g["n_s3"] > 0)).mean(), 3))
print("S2 matches per S1:", g["n_s2"].value_counts().sort("n_s2").rows()); print("S3 matches per S1:", g["n_s3"].value_counts().sort("n_s3").rows())

## 6. France (test only, no labels)

In [ ]:
for s in (1, 2, 3):
    print(f"test S{s} France:", *[f"{n}  |  {a}" for n, a in T[s].filter(pl.col("country") == "France").select("business_name", "business_address").sample(6, seed=5).rows()], sep="\n    ")

In [ ]:
# Non-ASCII in France is real French orthography (é, è, ç, ô) plus injected accents
{s: round(T[s].filter(pl.col("country") == "France")["business_name"].str.contains(r"[^\x00-\x7F]").mean(), 3) for s in (1, 2, 3)}